<a href="https://colab.research.google.com/github/swarupmca/adi/blob/main/Copy_of_mdim_rag_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG chatbot for MDI Murshidabad (mdim.ac.in)
Chromium (Playwright) crawl + PDF extraction → Hugging Face embeddings → ChromaDB → Llama via Ollama → Flask chat UI → ngrok public URL.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Add your ngrok token as a Colab secret named `NGROK_TOKEN` (key icon in the left sidebar), or paste it when prompted in Step 8. Free token: https://dashboard.ngrok.com/get-started/your-authtoken
3. Crawl only with the site owner's permission and respect robots.txt.

## Step 1: Install dependencies

In [ ]:
!pip install -q playwright beautifulsoup4 lxml trafilatura pypdf sentence-transformers chromadb ollama flask pyngrok nest_asyncio
!playwright install chromium
!playwright install-deps chromium

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
191.1 MiB [] 0% 393.6s191.1 MiB [] 0% 17.0s191.1 MiB [] 0% 11.4s191.1 MiB [] 0% 5.5s191.1 MiB [] 2% 3.2s191.1 MiB [] 2% 2.7s191.1 MiB [] 3% 2.4s191.1 MiB [] 4% 2.7s191.1 MiB [] 4% 2.8s191.1 MiB [] 5% 2.7s191.1 MiB [] 6% 2.6s191.1 MiB [] 6% 2.7s191.1 MiB [] 7% 2.5s191.1 MiB [] 8% 2.3s191.1 MiB [] 9% 2.2s191.1 MiB [] 10% 2.2s191.1 MiB [] 11% 2.2s191.1 MiB [] 12% 2.2s191.1 MiB [] 12% 2.4s191.1 MiB [] 13% 2.6s191.1 MiB [] 13% 2.7s191.1 MiB [] 13% 2.8s191.1 MiB [] 13% 2.9s191.1 MiB [] 14% 2.9s191.1 MiB [] 14% 2.8s191.1 MiB [] 15% 2.8s191.1 MiB [] 16% 2.7s191.1 MiB [] 17% 2.6s191.1 

## Step 2: Install and start Ollama, pull Llama

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, os

MODEL = "llama3.2:3b"   # use "llama3.1:8b" for better answers (fits on a T4)

subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)
!ollama pull {MODEL}

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



## Step 3: Config and robots.txt check

In [ ]:
import requests
from urllib.robotparser import RobotFileParser

BASE = "https://www.mdim.ac.in"
ALLOWED_HOSTS = {"www.mdim.ac.in", "mdim.ac.in"}   # excludes subdomains such as the journal site
INSTITUTE = "Management Development Institute Murshidabad (MDI Murshidabad / MDIM)"

MAX_PAGES = 200          # safety cap on HTML pages
MAX_PDFS = 40            # safety cap on PDF documents
MIN_PDF_YEAR = 2024      # skip PDFs uploaded before this year (old recruitment ads etc.); set 0 to keep all
DELAY_SECONDS = 1.0      # polite crawl delay
UA = {"User-Agent": "Mozilla/5.0 (compatible; MDIM-RAG-Bot)"}

rp = RobotFileParser()
rp.set_url(f"{BASE}/robots.txt")
rp.read()
print("robots.txt allows homepage:", rp.can_fetch("*", BASE + "/"))

robots.txt allows homepage: True


## Step 4: Crawl pages with headless Chromium

In [ ]:
import asyncio, nest_asyncio
from urllib.parse import urljoin, urlparse, urldefrag
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

nest_asyncio.apply()

SKIP_EXT = (".jpg", ".jpeg", ".png", ".gif", ".svg", ".webp", ".pdf", ".zip", ".doc", ".docx",
            ".xls", ".xlsx", ".ppt", ".pptx", ".css", ".js", ".xml", ".mp4")
BAD_PARTS = ("/wp-admin", "/wp-login", "/wp-json", "/xmlrpc", "/feed", "/wp-content/", "replytocom", "/author/")

def is_internal(u):
    return urlparse(u).netloc.lower() in ALLOWED_HOSTS

def is_page(u):
    return is_internal(u) and not u.lower().endswith(SKIP_EXT) and not any(b in u for b in BAD_PARTS)

def urls_from_sitemap(url, depth=0):
    """Reads a sitemap and follows nested sitemap indexes."""
    try:
        xml = requests.get(url, timeout=30, headers=UA).text
    except Exception:
        return []
    locs = [l.text.strip() for l in BeautifulSoup(xml, "xml").find_all("loc")]
    out = []
    for loc in locs:
        if loc.endswith(".xml") and depth < 2:
            out += urls_from_sitemap(loc, depth + 1)
        else:
            out.append(loc)
    return out

def discover_urls():
    urls = []
    # WordPress sitemap locations: Yoast / Rank Math / core
    for path in ("/sitemap_index.xml", "/sitemap.xml", "/wp-sitemap.xml"):
        urls = urls_from_sitemap(BASE + path)
        if urls:
            print("Sitemap found:", path)
            break
    if not urls:
        print("No sitemap found, will discover links starting from the homepage.")
        urls = [BASE + "/"]
    urls = [urldefrag(u)[0] for u in urls]
    urls = [u for u in dict.fromkeys(urls) if is_page(u)]
    return urls[:MAX_PAGES]

async def crawl(urls):
    docs, seen, seen_canonical, pdf_links = [], set(), set(), set()
    discover = len(urls) <= 1
    queue = list(urls)
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox", "--disable-dev-shm-usage"])
        page = await browser.new_page()
        while queue and len(docs) < MAX_PAGES:
            url = queue.pop(0)
            if url in seen or not rp.can_fetch("*", url):
                continue
            seen.add(url)
            try:
                await page.goto(url, wait_until="networkidle", timeout=45000)
                await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")  # trigger lazy content
                await page.wait_for_timeout(800)

                # WordPress exposes the same page as /slug/ and ?p=ID; keep only one copy
                canonical = await page.evaluate("document.querySelector('link[rel=canonical]')?.href || ''")
                if canonical and canonical in seen_canonical:
                    continue
                seen_canonical.add(canonical or url)

                html = await page.content()
                docs.append({"url": canonical or url, "title": await page.title(), "html": html})
                print(f"[{len(docs)}] {canonical or url}")

                # collect PDF links (brochures, fee structures, admission notices) and, if needed, page links
                hrefs = await page.eval_on_selector_all("a[href]", "els => els.map(e => e.href)")
                for h in hrefs:
                    h = urldefrag(h)[0]
                    if not is_internal(h):
                        continue
                    if h.lower().endswith(".pdf"):
                        pdf_links.add(h)
                    elif discover and is_page(h):
                        queue.append(h)
                await asyncio.sleep(DELAY_SECONDS)
            except Exception as e:
                print("failed:", url, str(e)[:80])
        await browser.close()
    return docs, pdf_links

urls = discover_urls()
print(len(urls), "URLs to crawl")
docs, pdf_links = await crawl(urls)
print("Crawled", len(docs), "pages;", len(pdf_links), "PDF links found")

Sitemap found: /sitemap_index.xml
200 URLs to crawl
[1] https://www.mdim.ac.in/
[2] https://www.mdim.ac.in/news/
[3] https://www.mdim.ac.in/alumni/
[4] https://www.mdim.ac.in/vision-mission/
[5] https://www.mdim.ac.in/social-responsibilities/
[6] https://www.mdim.ac.in/useful-links/
[7] https://www.mdim.ac.in/medical-assistance/
[8] https://www.mdim.ac.in/recreation-facilities/
[9] https://www.mdim.ac.in/students-hostel/
[10] https://www.mdim.ac.in/executive-hostel/
[11] https://www.mdim.ac.in/campus-amenities/
[12] https://www.mdim.ac.in/archives/
[13] https://www.mdim.ac.in/archives/tender-archives/
[14] https://www.mdim.ac.in/gallery/
[15] https://www.mdim.ac.in/career-with-us/
[16] https://www.mdim.ac.in/annual-events/
[17] https://www.mdim.ac.in/competition/
[18] https://www.mdim.ac.in/other-achievements/
[19] https://www.mdim.ac.in/mdi-advantage/
[20] https://www.mdim.ac.in/placement-process/
[21] https://www.mdim.ac.in/paper-submission-guidelines/
[22] https://www.mdim.ac.in/aca

## Step 5: Extract text from PDFs

In [ ]:
import io, re
from urllib.parse import unquote
from pypdf import PdfReader

def pdf_year(u):
    m = re.search(r"/uploads/(\d{4})/", u)
    return int(m.group(1)) if m else None

def fetch_pdfs(links):
    out = []
    for u in sorted(links):
        if len(out) >= MAX_PDFS:
            break
        y = pdf_year(u)
        if y and y < MIN_PDF_YEAR:
            continue
        if not rp.can_fetch("*", u):
            continue
        try:
            r = requests.get(u, timeout=60, headers=UA)
            if len(r.content) > 15_000_000:
                continue
            reader = PdfReader(io.BytesIO(r.content))
            text = "\n".join((pg.extract_text() or "") for pg in reader.pages)
            if len(text.strip()) > 200:
                out.append({"url": u, "title": unquote(u.split("/")[-1]), "text": text})
                print("PDF:", unquote(u.split("/")[-1]))
            time.sleep(0.5)
        except Exception as e:
            print("pdf failed:", u, str(e)[:60])
    return out

pdf_docs = fetch_pdfs(pdf_links)
print("Usable PDFs:", len(pdf_docs))

pdf failed: https://www.mdim.ac.in/wp-content/uploads/2024/01/Revised-fees-structure-PGDM-2024-26.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2024/02/2021-23_PGDM_Addmission_List_NBA.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2024/02/2022-24_PGDM_Addmission_List_NBA.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2024/02/2023-25_PGDM_Addmission_List_NBA.pdf Stream has ended unexpectedly


pdf failed: https://www.mdim.ac.in/wp-content/uploads/2024/05/AMDISA-CERTIFICATE.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2024/06/NBA_2425-2627.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2024/12/MDIM-Murshidabad-PGDM-Letter.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/01/GST-Certificate-NEW.pdf Stream has ended unexpectedly


pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/02/Fees-waiver-23pgdm.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/02/Fees-waiver-24pgdm.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/02/MDI-Murshidabad.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/03/Revised-Fee-Structure-for-PGDM-2025-27-Batch.pdf Stream has ended unexpectedly


pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/05/Education-Loan-Offer-Details.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/05/FPM-Fee-Structure-1.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/05/Online-PGDM-Fee-Structure-1.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/06/icici_bank_education_loan.pdf Stream has ended unexpectedly


pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/07/MDIM_DS_CIR_13_2025_Employee-Counselling-and-Grievance-Redressa-1.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/10/Academic_Calendar_FPM-2025-Batch.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2025/11/MDI-Gurgaon-Murshidabad_9x12_13-Nov-2025_251119_151728.pdf ('Connection aborted.', ConnectionResetError(104, 'Connectio
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2026/02/NE-Student-Scholarship-2026.pdf Stream has ended unexpectedly


pdf failed: https://www.mdim.ac.in/wp-content/uploads/2026/03/2026-28-fees-structure.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2026/04/NIRF-2026.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2026/05/SIP-Report-2025-27.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2026/07/Academic-Calender-PGDM-2025-27-for-AY-2026-27.pdf Stream has ended unexpectedly


pdf failed: https://www.mdim.ac.in/wp-content/uploads/2026/07/Academic-Calender-PGDM-2026-28-for-AY-2026-27.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2026/08/EOA-Report-26-27-MDI-Murshidabad.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2026/08/Final-Placement-Report-2024-26_17-8-2026.pdf Stream has ended unexpectedly
pdf failed: https://www.mdim.ac.in/wp-content/uploads/2026/10/AACSB-Membership-Certificate-Document.pdf Stream has ended unexpectedly
Usable PDFs: 0


## Step 6: Clean, chunk, embed, and store

In [ ]:
import trafilatura
from sentence_transformers import SentenceTransformer
import chromadb

def clean_html(html):
    text = trafilatura.extract(html, include_tables=True, include_links=False)
    if not text:
        soup = BeautifulSoup(html, "lxml")
        for t in soup(["nav", "header", "footer", "script", "style", "noscript"]):
            t.decompose()
        text = soup.get_text("\n")
    return re.sub(r"\n{3,}", "\n\n", text).strip()

clean_docs = []
for d in docs:
    t = clean_html(d["html"])
    if len(t) > 200:
        clean_docs.append({"url": d["url"], "title": d["title"], "text": t})
clean_docs += pdf_docs
print("Usable documents:", len(clean_docs))

def chunk(text, size=1200, overlap=150):
    paras, chunks, cur = [p.strip() for p in text.split("\n") if p.strip()], [], ""
    for p in paras:
        if len(cur) + len(p) + 1 <= size:
            cur += ("\n" if cur else "") + p
        else:
            if cur: chunks.append(cur)
            cur = (cur[-overlap:] + "\n" if cur else "") + p
    if cur: chunks.append(cur)
    return chunks

chunks, metas, ids = [], [], []
for d in clean_docs:
    for j, c in enumerate(chunk(d["text"])):
        chunks.append(f'{d["title"]}\n{c}')
        metas.append({"url": d["url"], "title": d["title"]})
        ids.append(f'{d["url"]}#{j}')
print("Chunks:", len(chunks))

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
embeddings = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True, batch_size=64)

client = chromadb.PersistentClient(path="/content/mdim_db")
try: client.delete_collection("mdim")
except Exception: pass
col = client.create_collection("mdim", metadata={"hnsw:space": "cosine"})
for i in range(0, len(ids), 500):
    col.add(ids=ids[i:i+500], documents=chunks[i:i+500],
            metadatas=metas[i:i+500], embeddings=embeddings[i:i+500].tolist())
print("Stored", col.count(), "chunks")

Usable documents: 200
Chunks: 203


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Stored 203 chunks


## Step 7: Retrieval and generation

In [ ]:
import ollama

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # recommended for bge retrieval

def retrieve(question, k=5):
    q = embedder.encode([QUERY_PREFIX + question], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q, n_results=k)
    return list(zip(res["documents"][0], res["metadatas"][0]))

def answer(question, k=5):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{i+1}] ({m['url']})\n{doc}" for i, (doc, m) in enumerate(hits))
    prompt = f"""You are the information assistant for {INSTITUTE}.
Answer ONLY using the context below. If the answer is not in the context,
say you don't have that information and suggest contacting the institute through www.mdim.ac.in.
Be concise and cite sources like [1], [2].
For fees, deadlines, seats and eligibility, remind the user to confirm on the official website
because these change every admission cycle.

Context:
{context}

Question: {question}
Answer:"""
    r = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}],
                    options={"temperature": 0.2})
    sources = list(dict.fromkeys(m["url"] for _, m in hits))
    return r["message"]["content"], sources

a, s = answer("What programmes does MDI Murshidabad offer?")
print(a, "\n", s)

MDI Murshidabad offers future-ready programs that empower students to grow as capable managers and confident leaders. Choose from programs such as:

[1] (https://www.mdim.ac.in/)
Home - MDI Murshidabad
...future-ready programs that empower students to grow as capable managers and confident leaders. Gain practical knowledge, industry exposure, and the skills needed to become responsible business professionals.

[4] (https://www.mdim.ac.in/)
Home - MDI Murshidabad
...future-ready programs that are designed to shape future-ready management professionals.

[5] (https://www.mdim.ac.in/announcements/mdi-murshidabad-invites-sealed-tenders-from-reputed-vendors-for-campus-wi-fi-of-the-institute/)
...future-ready programs that are designed to equip students with the skills needed to become responsible business professionals.

Please note that the exact programs offered by MDI Murshidabad may be subject to change, and it's recommended to check the official website for the most up-to-date informat

## Step 8: Flask chat UI + ngrok public URL

In [ ]:
from flask import Flask, request, jsonify, Response
from pyngrok import ngrok
import threading, getpass

INDEX_HTML = """<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>MDI Murshidabad Assistant</title>
<style>
  *{box-sizing:border-box} body{margin:0;font-family:system-ui,sans-serif;background:#f3f4f6;color:#111}
  .wrap{max-width:780px;margin:0 auto;height:100vh;display:flex;flex-direction:column}
  header{padding:16px 20px;background:#7f1d1d;color:#fff}
  header b{font-size:17px} header small{display:block;opacity:.85;margin-top:2px}
  #chat{flex:1;overflow-y:auto;padding:20px;display:flex;flex-direction:column;gap:12px}
  .msg{max-width:85%;padding:10px 14px;border-radius:14px;line-height:1.5;white-space:pre-wrap;word-wrap:break-word}
  .user{align-self:flex-end;background:#7f1d1d;color:#fff}
  .bot{align-self:flex-start;background:#fff;border:1px solid #e5e7eb}
  .src{margin-top:8px;font-size:12px;color:#555}
  .src a{color:#7f1d1d;display:block;word-break:break-all}
  .chips{display:flex;flex-wrap:wrap;gap:8px;padding:0 20px 10px}
  .chip{border:1px solid #d1d5db;background:#fff;border-radius:999px;padding:6px 12px;font-size:13px;cursor:pointer;color:#374151}
  .chip:hover{background:#fef2f2;border-color:#7f1d1d}
  form{display:flex;gap:8px;padding:14px;background:#fff;border-top:1px solid #e5e7eb}
  input{flex:1;padding:12px;border:1px solid #d1d5db;border-radius:10px;font-size:15px}
  button.send{padding:0 20px;border:0;border-radius:10px;background:#7f1d1d;color:#fff;font-size:15px;cursor:pointer}
  button:disabled{opacity:.5;cursor:wait}
</style></head><body>
<div class="wrap">
  <header><b>MDI Murshidabad Assistant</b><small>Answers are based on content from mdim.ac.in</small></header>
  <div id="chat"><div class="msg bot">Hello! Ask me about programmes, admissions, MDPs, or the campus.</div></div>
  <div class="chips" id="chips">
    <button class="chip">What programmes does MDIM offer?</button>
    <button class="chip">How do I apply for the BBA programme?</button>
    <button class="chip">Tell me about the FPM programme</button>
    <button class="chip">What are the MDP/FDP programmes?</button>
  </div>
  <form id="f"><input id="q" placeholder="Type your question..." autocomplete="off" required>
  <button class="send" id="b">Send</button></form>
</div>
<script>
const chat=document.getElementById('chat'), f=document.getElementById('f'),
      q=document.getElementById('q'), b=document.getElementById('b');
function add(text, cls, sources){
  const d=document.createElement('div'); d.className='msg '+cls; d.textContent=text;
  if(sources && sources.length){
    const s=document.createElement('div'); s.className='src'; s.textContent='Sources:';
    sources.forEach(u=>{const a=document.createElement('a'); a.href=u; a.target='_blank'; a.textContent=u; s.appendChild(a);});
    d.appendChild(s);
  }
  chat.appendChild(d); chat.scrollTop=chat.scrollHeight; return d;
}
async function send(question){
  add(question,'user'); b.disabled=true;
  const wait=add('Thinking...','bot');
  try{
    const r=await fetch('/ask',{method:'POST',
      headers:{'Content-Type':'application/json','ngrok-skip-browser-warning':'1'},
      body:JSON.stringify({question})});
    const data=await r.json(); wait.remove();
    add(data.answer || data.error || 'No response','bot',data.sources);
  }catch(err){ wait.remove(); add('Error: '+err,'bot'); }
  b.disabled=false; q.focus();
}
f.addEventListener('submit', e=>{
  e.preventDefault();
  const t=q.value.trim(); if(!t) return;
  q.value=''; send(t);
});
document.querySelectorAll('.chip').forEach(c=>c.addEventListener('click',()=>{
  document.getElementById('chips').style.display='none'; send(c.textContent);
}));
</script></body></html>"""

app = Flask(__name__)

@app.get("/")
def index():
    return Response(INDEX_HTML, mimetype="text/html")

@app.post("/ask")
def ask():
    try:
        question = (request.get_json(force=True).get("question") or "").strip()
        if not question:
            return jsonify({"error": "Empty question"}), 400
        a, sources = answer(question)
        return jsonify({"answer": a, "sources": sources})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.get("/health")
def health():
    return jsonify({"status": "ok", "chunks": col.count(), "model": MODEL})

try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
except Exception:
    NGROK_TOKEN = getpass.getpass("Paste your ngrok authtoken: ")

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()                                  # close old tunnels on re-run
tunnel = ngrok.connect(5000)
print("Public URL:", tunnel.public_url)

threading.Thread(target=lambda: app.run(host="0.0.0.0", port=5000, use_reloader=False), daemon=True).start()

Paste your ngrok authtoken: ··········
Public URL: https://slobbery-wisdom-vowel.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000


## Notes
- Open the **Public URL** printed above. On ngrok's free plan you may see a one-time "Visit Site" warning page.
- The Colab session must stay alive for the link to work. Re-running Step 8 gives a new URL.
- WordPress serves the same page as `/slug/` and `?p=ID`; the crawler keeps one copy using the canonical link.
- Old PDFs (such as past recruitment ads) are skipped by `MIN_PDF_YEAR`, so the bot doesn't quote outdated notices. Lower it to include them.
- If the site is redesigned or content changes, re-run Steps 4 to 6 to rebuild the index.
- Embedding the chat on mdim.ac.in itself needs CORS (`pip install flask-cors`, then `CORS(app)`) and a permanent host instead of Colab.